# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jericho-Ram/FlyRank-Internship-ML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue ranks the same population from Weeks 5-6 (26,604 rows, base rate 0.611) using two
independent signals side by side, instead of collapsing them into one black-box number: the
Week-4 hand rule (missed clicks against tier-expected CTR) and the Week-5 RandomForest,
re-scored here out-of-fold on the exact grouped folds from Week 6 (permissive OOF AUC 0.662,
strict OOF AUC 0.628 -- pooled across all folds, not the per-fold mean Week 6 quotes, so the
numbers move slightly but tell the same story).

Four tiers, in priority order:

1. **Refresh now** (5,396 rows) -- rule and model agree the page is underperforming. Sorted
   within tier by the rule's missed-click share, since that number is tied to an actual
   measured traffic gap, not just a probability.
2. **Investigate -- rule silent, model flags it** (11,276 rows) -- the rule sees no CTR gap
   (usually because expected clicks are too small to score, or there genuinely is no gap), but
   the model puts p(decline) >= 0.6. This is the subset Week 6 measured the model adding real
   value on top of the rule; here it reads a 70.2% true-decline rate, close to Week 6's 60.1%
   figure on the full rule-silent population -- this narrower p >= 0.6 cut is more selective,
   which is why it reads higher. Flagged for investigation, not automatic action -- see
   Section 3.
3. **CTR gap only, model unsure** (1,730 rows) -- the rule found a real, measured click gap
   against tier-expected CTR even though the model's decline probability is under 0.5. Kept in
   the queue because the rule answers a different question (are we getting the clicks this
   position should get) than the model (is this page trending down) -- a page can lag its
   position's expected CTR while its 30-day trend still looks stable.
4. **Monitor** (8,202 rows) -- neither signal flags a problem right now.

Reason codes (multi-label, attached to every row in the exported queue): `rule_and_model_agree`,
`model_flags_rule_silent`, `rule_flags_model_unsure`, `high_confidence_model` (p >= 0.7),
`window_reliant_caveat` (Section 4), `top_visible_tier`, `outside_rule_scope` (Section 4), and
`recently_updated_flag` (Section 5) -- a human scanning the queue reads the codes, not the model
internals, to know why a row landed where it did.

In [1]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/Jericho-Ram/FlyRank-Internship-ML"
    REPO_DIR = "FlyRank-Internship-ML"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if not IN_COLAB and os.getcwd().replace("\\", "/").endswith("work/notebooks"):
    os.chdir("../..")

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import sklearn  # noqa: E402

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
np.random.seed(SEED)

if os.getcwd().replace("\\", "/").endswith("work/notebooks"):
    os.chdir("../..")

CSV_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(CSV_PATH), "starter CSV not found -- are you at the repo root?"
df = pd.read_csv(CSV_PATH)

# --- Rebuild the Week-5/6 population, label, feature sets ------------------
ranked = df["avg_position"] > 0
can_decline = df["impressions_prev_30d"] > 0
pop = df[ranked & can_decline].reset_index(drop=True).copy()
y = pop["trend_direction"].str.lower().eq("down").to_numpy()
groups = pop["client_id"].to_numpy()

LEAK_COLS = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d"]
WINDOW_COLS = ["clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"]
DROP_ALWAYS = ["content_id", "client_id"] + LEAK_COLS
FEATURES = [c for c in pop.columns if c not in DROP_ALWAYS]
STRICT_FEATURES = [c for c in FEATURES if c not in WINDOW_COLS]

print(f"population: {len(pop):,} rows | permissive features: {len(FEATURES)} | base rate: {y.mean():.3f}")

# --- Week-4 rule, re-scored on this population (same as w05) ---------------
tier_stats = df[ranked].groupby("position_tier").agg(ti=("impressions_90d", "sum"), tc=("clicks_90d", "sum"))
TIER_CTR = tier_stats["tc"] / tier_stats["ti"]
EXPECTED_CLICKS_FLOOR = 5.0
IN_SCOPE_TIERS = ["page_1", "striking", "page_3_5", "top_3"]
pop["tier_expected_ctr"] = pop["position_tier"].map(TIER_CTR)
pop["expected_clicks"] = pop["impressions_90d"] * pop["tier_expected_ctr"]
pop["missed_clicks"] = pop["expected_clicks"] - pop["clicks_90d"]
_in_scope = pop["position_tier"].isin(IN_SCOPE_TIERS)
_scorable = _in_scope & (pop["expected_clicks"] >= EXPECTED_CLICKS_FLOOR)
_fires = _scorable & (pop["missed_clicks"] > 0)
pop["baseline_score"] = np.where(_fires, pop["missed_clicks"] / pop["expected_clicks"], 0.0)
pop["rule_in_scope"] = _in_scope


def build_pipe(model, cols):
    num = [c for c in cols if pd.api.types.is_numeric_dtype(pop[c])]
    cat = [c for c in cols if c not in num]
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat),
    ])
    return Pipeline([("pre", pre), ("model", model)])


GKF = GroupKFold(n_splits=5)
folds = list(GKF.split(pop, y, groups))


def cv_oof(cols):
    oof = np.zeros(len(pop))
    for tr, te in folds:
        rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
        pipe = build_pipe(rf, cols)
        pipe.fit(pop[cols].iloc[tr], y[tr])
        oof[te] = pipe.predict_proba(pop[cols].iloc[te])[:, 1]
    return oof


# --- Honest out-of-fold model scores (same model as w05/w06) ---------------
p_permissive = cv_oof(FEATURES)
p_strict = cv_oof(STRICT_FEATURES)
print(f"\nout-of-fold ROC AUC, permissive : {roc_auc_score(y, p_permissive):.3f}")
print(f"out-of-fold ROC AUC, strict     : {roc_auc_score(y, p_strict):.3f}")
print("(pooled OOF AUC, not the per-fold mean quoted in w06 -- same model, same folds, different aggregation)")

# --- Build the ranked queue: rule x model agreement, plus reason codes -----
q = pop[["content_id", "client_id", "position_tier", "rule_in_scope", "baseline_score",
         "impressions_90d", "clicks_90d", "avg_position", "content_age_days",
         "days_since_last_update", "content_type"]].copy()
q["truth_declining"] = y
q["p_decline"] = p_permissive
q["p_decline_strict"] = p_strict
q["window_reliance"] = q["p_decline"] - q["p_decline_strict"]

VISIBLE_TIERS = ["page_1", "top_3", "striking"]
q["rule_flags"] = q["baseline_score"] > 0.0
q["model_flags"] = q["p_decline"] >= 0.5
q["window_reliant"] = q["window_reliance"] >= 0.05
q["top_visible"] = q["position_tier"].isin(VISIBLE_TIERS)
q["high_confidence_model"] = q["p_decline"] >= 0.7
q["recently_updated"] = q["days_since_last_update"] < 14


def action_tier(row):
    if row["rule_flags"] and row["model_flags"]:
        return "1_refresh_now"
    if (not row["rule_flags"]) and row["p_decline"] >= 0.6:
        return "2_investigate_rule_silent"
    if row["rule_flags"] and not row["model_flags"]:
        return "3_ctr_gap_only"
    return "4_monitor"


q["action_tier"] = q.apply(action_tier, axis=1)


def reason_codes(row):
    codes = []
    if row["rule_flags"] and row["model_flags"]:
        codes.append("rule_and_model_agree")
    if (not row["rule_flags"]) and row["p_decline"] >= 0.6:
        codes.append("model_flags_rule_silent")
    if row["rule_flags"] and not row["model_flags"]:
        codes.append("rule_flags_model_unsure")
    if row["high_confidence_model"]:
        codes.append("high_confidence_model")
    if row["window_reliant"]:
        codes.append("window_reliant_caveat")
    if row["top_visible"]:
        codes.append("top_visible_tier")
    if not row["rule_in_scope"]:
        codes.append("outside_rule_scope")
    if row["recently_updated"]:
        codes.append("recently_updated_flag")
    return "|".join(codes)


q["reason_codes"] = q.apply(reason_codes, axis=1)
q_sorted = q.sort_values(["action_tier", "baseline_score"], ascending=[True, False]).reset_index(drop=True)
q_sorted.insert(0, "rank", np.arange(1, len(q_sorted) + 1))

print("\n--- action tier counts ---")
print(q_sorted["action_tier"].value_counts().sort_index().to_string())

print("\n--- top 5 rows ---")
print(q_sorted[["rank", "content_id", "action_tier", "baseline_score", "p_decline", "reason_codes"]]
      .head(5).to_string(index=False))

population: 26,604 rows | permissive features: 38 | base rate: 0.611

out-of-fold ROC AUC, permissive : 0.662
out-of-fold ROC AUC, strict     : 0.628
(pooled OOF AUC, not the per-fold mean quoted in w06 -- same model, same folds, different aggregation)

--- action tier counts ---
action_tier
1_refresh_now                 5396
2_investigate_rule_silent    11276
3_ctr_gap_only                1730
4_monitor                     8202

--- top 5 rows ---
 rank           content_id   action_tier  baseline_score  p_decline                                                                      reason_codes
    1 content_9983d31c53cb 1_refresh_now             1.0   0.735657 rule_and_model_agree|high_confidence_model|top_visible_tier|recently_updated_flag
    2 content_e9785b5bd320 1_refresh_now             1.0   0.663164                                             rule_and_model_agree|top_visible_tier
    3 content_bb2273ff6eed 1_refresh_now             1.0   0.774652                       rule_an

## 2. Archetypes -> action mapping

*What recurring kinds of page exist here, and what each kind deserves.*

**What this is, and what it is not.** This is **metric clustering** -- K-Means (k=5) over nine
scaled performance and metadata columns. This dataset holds no article text, so this cannot be
and is not semantic clustering. The archetype names are labels I attached after reading the
profiles, not categories the algorithm discovered.

**One column is deliberately excluded, and the reason matters.** `days_since_last_update` is
**not** a clustering feature here. It carries 57 distinct values across 30,000 rows, and five of
them cover 87.8% -- two values alone (`20` and `104`) account for roughly 68%. Those are batch
write timestamps, not per-page freshness, which is the same limitation already recorded on the
deployed paper.

I built a first version of this section *with* that column included. It produced a clean-looking
sixth-of-the-population archetype I was about to name "stale long-form with reach," on the
strength of a median 104 days since update and 97.7% of its rows sitting above 90 days. That
`104` is one of the two batch values. When the column is dropped, **that archetype does not
shrink -- it disappears entirely**, redistributing into the visible and low-visibility groups.
It was an artifact of which batch a row was stamped in, wearing the name of a content insight.
I'm recording that here rather than quietly shipping the corrected version, because the failure
is more instructive than the fix.

**The five profiles, and the action each earns:**

| Name | Share | Decline rate | Profile (medians) | Action |
|---|---|---|---|---|
| **Visible performers** | 29.1% | 0.547 | 8,699 impressions, 23 clicks, CTR 0.29, position 7.9, the only group with real engagement (2.33) | **Protect** |
| **Deep low-demand** | 31.4% | 0.537 | 530 impressions, 0 clicks, position 19.0, oldest group at 390 days | **Merge or prune** |
| **Young unproven** | 29.6% | **0.741** | 559 impressions, 1 click, position 11.7, youngest at 138 days | **Wait** |
| **Low-visibility mid-age** | 9.4% | 0.663 | 82 impressions, 0 clicks, position 9.5, 180 days | **Investigate** |
| **Micro-volume** | 0.5% | 0.393 | 6 impressions; n=140 | **Exclude** |

**How weak the structure is.** Silhouette across k=3..6 runs 0.175 to 0.194 -- k=5 scores 0.179
and is not even the local best (k=6 is). All are well under 0.2: weak separation. The inventory
is closer to a continuum than to five distinct populations. I keep k=5 because the profiles are
readable and map to distinct actions, **not** because the data says five is right.

**The mapping earns its place by disagreeing with the decline rate.** Young unproven has the
highest decline rate in the population (0.741) and is the *worst* place to spend editor time: it
holds 5.8% of impressions and 9.7% of the recoverable missed clicks, and its pages are ~138 days
old -- young enough that a downward 30-day window more likely catches a page still finding its
level than a page decaying. Sorting the queue by "most declining" sends an editor straight there.

**What this section cannot tell you.** With no usable freshness column, these archetypes cannot
identify a "stale but still valuable" group -- the very category a refresh product most wants.
That target has to be defined from the measured CTR gap and model agreement in Section 1, not
from page age or update recency. That is a real limit of this extract, not an oversight here.


In [2]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# --- why days_since_last_update is NOT a feature below ---
_d = df["days_since_last_update"]
print(f"days_since_last_update: {_d.nunique()} distinct values over {len(df):,} rows; "
      f"top 5 cover {_d.value_counts().head(5).sum() / len(df):.1%}")
print(f"  the two largest: {_d.value_counts().head(2).to_dict()}")
print(f"content_age_days      : {df['content_age_days'].nunique()} distinct values; "
      f"top 5 cover {df['content_age_days'].value_counts().head(5).sum() / len(df):.1%}")
print("-> update recency is batch-stamped and excluded; page age is usable and kept\n")

ARCHETYPE_FEATURES = [
    "impressions_90d", "clicks_90d", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "content_age_days", "word_count", "search_volume",
]
LOG_COLS = ["impressions_90d", "clicks_90d", "search_volume", "word_count", "content_age_days"]

A = pop[ARCHETYPE_FEATURES].copy()
for c in LOG_COLS:                      # heavy-tailed -> log1p so no one column dominates
    A[c] = np.log1p(A[c].clip(lower=0))
A = A.fillna(A.median())
Xs = StandardScaler().fit_transform(A)

_sub = np.random.default_rng(SEED).choice(len(Xs), 5000, replace=False)
sil_by_k = {}
print("silhouette by k (5,000-row sample):")
for k in [3, 4, 5, 6]:
    _lab = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(Xs)
    sil_by_k[k] = float(silhouette_score(Xs[_sub], _lab[_sub]))
    print(f"  k={k}: {sil_by_k[k]:.3f}")
print("all < 0.2 -- weak separation; k=5 is chosen for readability, not because it scores best\n")

pop["archetype_id"] = KMeans(n_clusters=5, n_init=10, random_state=SEED).fit_predict(Xs)

_raw = pop.groupby("archetype_id").agg(
    med_impr=("impressions_90d", "median"), med_clicks=("clicks_90d", "median"),
    med_pos=("avg_position", "median"), med_age=("content_age_days", "median"),
    n=("content_id", "size"))

def name_cluster(r):
    """Deterministic naming from the profile, so names survive a re-run or a version bump."""
    if r.med_impr < 50:        return "micro_volume"
    if r.med_clicks >= 10:     return "visible_performers"
    if r.med_pos >= 15:        return "deep_low_demand"
    if r.med_age < 160:        return "young_unproven"
    return "low_visibility_midage"

ARCHETYPE_NAMES = {i: name_cluster(r) for i, r in _raw.iterrows()}
assert len(set(ARCHETYPE_NAMES.values())) == 5, \
    f"naming rules collided -- profiles shifted, re-read them before trusting: {ARCHETYPE_NAMES}"

ARCHETYPE_ACTIONS = {
    "visible_performers": "protect",
    "deep_low_demand": "merge_or_prune",
    "young_unproven": "wait",
    "low_visibility_midage": "investigate",
    "micro_volume": "exclude",
}
pop["archetype"] = pop["archetype_id"].map(ARCHETYPE_NAMES)
pop["archetype_action"] = pop["archetype"].map(ARCHETYPE_ACTIONS)

archetype_profile = pop.groupby("archetype").agg(
    n=("content_id", "size"),
    decline_rate=("trend_direction", lambda s: s.str.lower().eq("down").mean()),
    med_impressions=("impressions_90d", "median"),
    med_clicks=("clicks_90d", "median"),
    med_position=("avg_position", "median"),
    med_age_days=("content_age_days", "median"),
    med_word_count=("word_count", "median"),
    med_engagement=("engagement_rate", "median"),
    total_impressions=("impressions_90d", "sum"),
    recoverable_missed_clicks=("missed_clicks", lambda s: s[s > 0].sum()),
).sort_values("recoverable_missed_clicks", ascending=False)
archetype_profile["share_of_pages"] = archetype_profile["n"] / len(pop)
archetype_profile["share_of_impressions"] = (
    archetype_profile["total_impressions"] / archetype_profile["total_impressions"].sum())
archetype_profile["share_of_missed_clicks"] = (
    archetype_profile["recoverable_missed_clicks"] / archetype_profile["recoverable_missed_clicks"].sum())
archetype_profile["action"] = archetype_profile.index.map(ARCHETYPE_ACTIONS)

print("=== archetype profiles (k=5, metric clustering -- NOT semantic) ===")
print(archetype_profile[[
    "n", "share_of_pages", "decline_rate", "med_impressions", "med_clicks", "med_position",
    "med_age_days", "med_engagement", "share_of_impressions", "share_of_missed_clicks",
    "action"]].round(3).to_string())

q_sorted = q_sorted.merge(
    pop[["content_id", "archetype", "archetype_action"]], on="content_id", how="left")

print("\n=== archetype x action tier ===")
print(pd.crosstab(q_sorted["archetype"], q_sorted["action_tier"]).to_string())


days_since_last_update: 57 distinct values over 30,000 rows; top 5 cover 87.8%
  the two largest: {20: 11573, 104: 8773}
content_age_days      : 225 distinct values; top 5 cover 12.1%
-> update recency is batch-stamped and excluded; page age is usable and kept

silhouette by k (5,000-row sample):
  k=3: 0.175
  k=4: 0.187
  k=5: 0.179
  k=6: 0.194
all < 0.2 -- weak separation; k=5 is chosen for readability, not because it scores best

=== archetype profiles (k=5, metric clustering -- NOT semantic) ===
                          n  share_of_pages  decline_rate  med_impressions  med_clicks  med_position  med_age_days  med_engagement  share_of_impressions  share_of_missed_clicks          action
archetype                                                                                                                                                                                       
visible_performers     7740           0.291         0.547           8699.0        23.0          7.90       

## 3. The decay finding -- and the column that cannot support one

*What this extract does and does not say about content going stale.*

A refresh product rests on the premise that pages decay as they age and go stale, so the stale
ones are the ones to fix. This extract can test half of that premise and cannot test the other
half at all. Both halves are worth stating.

### The half it cannot test

`days_since_last_update` is the only column describing update recency, and it is unusable for
the purpose: 57 distinct values over 30,000 rows, top five covering 87.8%, with `20` and `104`
alone accounting for ~68%. Those are batch write timestamps.

So the flat-looking relationship between `freshness_tier` and the label below is **not** evidence
that staleness fails to predict decline. It is evidence that **this column does not measure
staleness**, which means no staleness conclusion can be drawn from it in either direction. The
bins are also unusable on their own terms -- 17,448 rows in one and 133 in another.

| Freshness tier | n | Declining rate |
|---|---|---|
| 0-30 days | 17,448 | 0.600 |
| 31-90 days | 167 | 0.617 |
| 91-180 days | 8,856 | 0.633 |
| 181+ days | 133 | 0.617 |

I include the table only to show the artifact, not to draw from it. Point-biserial r is +0.023.

### The half it can test, and the answer is backwards

`content_age_days` is a usable column -- 225 distinct values, top five covering 12.1%. And it
runs opposite to the premise:

| Age tier | n | Declining rate |
|---|---|---|
| 31-90 days | 462 | **0.712** |
| 91-180 days | 10,428 | 0.707 |
| 181-365 days | 9,607 | 0.609 |
| 365+ days | 6,107 | **0.444** |

Point-biserial r = **-0.203**, monotonic across all four tiers. Older pages decline *less*.

**How I read that, carefully.** The most plausible reading is volatility rather than decay: a
page around 120 days old is still settling into a position, and a 30-day window is more likely
to catch it mid-swing. That is consistent with the young-unproven archetype in Section 2 --
youngest group, highest decline rate, negligible reach.

**The confound I cannot rule out, and it is serious.** This population requires
`impressions_prev_30d > 0`. Old pages that already decayed to zero impressions were filtered out
before anything was measured. "Old pages decline less" may substantially be **survivorship** --
the old pages still here are the survivors, and the ones that decayed are invisible. I cannot
separate genuine stabilisation from survivorship with this extract and I am not claiming which
it is.

**What none of this licenses.** This extract contains no refresh outcomes, so it cannot evaluate
refreshing at all. The operational consequence is narrow and specific: **do not sort this queue
by age or by update recency.** One column can't measure what it claims to, and the other points
the wrong way for reasons that may be an artifact of who survived to be measured.


In [3]:
print("=== the column that cannot support a staleness claim ===")
_vc = df["days_since_last_update"].value_counts()
print(f"distinct values : {df['days_since_last_update'].nunique()} over {len(df):,} rows")
print(f"top 5 cover     : {_vc.head(5).sum() / len(df):.1%}")
print(f"two largest     : {_vc.head(2).to_dict()}  <- batch write timestamps\n")

fresh_decay = pop.groupby("freshness_tier").agg(
    n=("content_id", "size"),
    declining_rate=("trend_direction", lambda s: s.str.lower().eq("down").mean()),
    med_days_since_update=("days_since_last_update", "median"),
).sort_values("med_days_since_update")
print("declining rate by freshness tier (shown as artifact, not as finding):")
print(fresh_decay.round(3).to_string())

print("\n=== the column that can: content_age_days ===")
print(f"distinct values : {df['content_age_days'].nunique()}; "
      f"top 5 cover {df['content_age_days'].value_counts().head(5).sum() / len(df):.1%}\n")
age_decay = pop.groupby("age_tier").agg(
    n=("content_id", "size"),
    declining_rate=("trend_direction", lambda s: s.str.lower().eq("down").mean()),
    med_age_days=("content_age_days", "median"),
).sort_values("med_age_days")
print(age_decay.round(3).to_string())

print("\n=== point-biserial correlation with the declining label ===")
decay_corr = {}
for c in ["content_age_days", "days_since_last_update", "avg_position", "ctr"]:
    r = float(np.corrcoef(pop[c].fillna(pop[c].median()), y.astype(float))[0, 1])
    decay_corr[c] = round(r, 4)
    print(f"  {c:24s} r = {r:+.4f}")

print(f"\nspread across age tiers       : "
      f"{age_decay['declining_rate'].max() - age_decay['declining_rate'].min():.3f}")
print("\nsurvivorship caveat: population requires impressions_prev_30d > 0, so pages that")
print("already decayed to zero impressions are absent -- the age effect cannot be cleanly")
print("separated from survivorship with this extract.")


=== the column that cannot support a staleness claim ===
distinct values : 57 over 30,000 rows
top 5 cover     : 87.8%
two largest     : {20: 11573, 104: 8773}  <- batch write timestamps

declining rate by freshness tier (shown as artifact, not as finding):
                    n  declining_rate  med_days_since_update
freshness_tier                                              
0-30            17448           0.600                   20.0
31-90             167           0.617                   41.0
91-180           8856           0.633                  104.0
181+              133           0.617                  211.0

=== the column that can: content_age_days ===
distinct values : 225; top 5 cover 12.1%

              n  declining_rate  med_age_days
age_tier                                     
31-90       462           0.712          90.0
91-180    10428           0.707         125.0
181-365    9607           0.609         286.0
365+       6107           0.444         463.0

=== point-

## 4. Intended use and limits

*Who uses this, for what -- and where it stops being valid.*

**Who:** a content/SEO editor deciding what to open first this sprint, across the clients in
this dataset. It's a triage queue, not an auto-publish pipeline -- see Section 5.

**Where it stops being valid:**

- **New clients.** This population covers 31 clients; the smallest has 2 rows, the largest
  is 26.2% of the population. Week 6 measured a 0.122 AUC gap between grouped and random
  splits on this exact model -- meaning a real share of its skill is client-specific pattern
  memorization. For a brand-new client not in this training population, only the
  non-memorized share of that skill should be assumed to carry over; the honest floor is
  closer to the strict OOF AUC than the permissive one.
- **Rows outside the rule's scope.** A block of rows sits in the `deep` position tier, which
  the Week-4 rule never evaluates (it only scores `page_1`, `top_3`, `striking`, `page_3_5`).
  Some of those still got a model-driven action-tier placement (`outside_rule_scope` in reason
  codes) -- read those as model-only opinions with no rule cross-check, not as agreed findings.
- **Window-reliant rows.** A meaningful share of the "refresh now" tier carries the
  `window_reliant_caveat` -- Week 6's Test E found 0.034 AUC is carried by columns
  (`clicks_last_30d`, `sessions_last_30d`, etc.) that share a time window with the label. Their
  rule score is unaffected (it doesn't use those columns), but their model score is partly
  riding on same-window correlation rather than a leading indicator.
- **Snapshot, not forecast.** Both scores describe the current 90/30-day windows in this
  extract. Neither claims a page will keep declining, only that it currently looks like the
  pages that historically did (model) or is currently under-clicking its position (rule).

In [4]:
n_clients = pop["client_id"].nunique()
byc = pop.groupby("client_id").size().sort_values()
print(f"clients in this population : {n_clients}")
print(f"smallest client, rows      : {byc.iloc[0]} ({byc.index[0]})")
print(f"largest client, rows       : {byc.iloc[-1]} ({byc.index[-1]}), {byc.iloc[-1] / len(pop) * 100:.1f}% of population")
print(f"rows outside rule's scope ('deep' tier, rule never evaluates) : {(~pop['rule_in_scope']).sum():,}")

t2 = q_sorted[q_sorted["action_tier"] == "2_investigate_rule_silent"]
print(f"tier 2 (rule silent, model flags) n={len(t2):,}, true declining rate={t2['truth_declining'].mean():.3f}")

t1 = q_sorted[q_sorted["action_tier"] == "1_refresh_now"]
print(f"tier 1 window-reliant share : {t1['window_reliant'].mean():.3f} ({t1['window_reliant'].sum():,} of {len(t1):,})")

clients in this population : 31
smallest client, rows      : 2 (client_1a6562590e)
largest client, rows       : 6981 (client_19581e27de), 26.2% of population
rows outside rule's scope ('deep' tier, rule never evaluates) : 1,136
tier 2 (rule silent, model flags) n=11,276, true declining rate=0.702
tier 1 window-reliant share : 0.229 (1,233 of 5,396)


## 5. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on tier 1 or 2:**
- Confirm the page still exists and isn't already superseded or redirected -- the dataset has
  no "deleted" flag.
- Check `recently_updated_flag`: a real slice of tier-1 rows were updated fewer than 14 days
  ago. A page just touched and already flagged again is either a false signal from
  update-day measurement noise, or a real sign the last update didn't work -- a human needs to
  tell which; the model can't.
- Check `outside_rule_scope`: some rows get a tier from the model alone, with no rule
  cross-check (Section 4). Read these as leads, not confirmed findings.
- For any client with thin representation in this population, treat tier assignments as
  low-confidence -- there isn't enough of that client's data in these folds for the grouped-CV
  numbers to say much about them specifically.

**No-go -- never automate:**
- Never auto-publish or auto-edit content from this queue. The output is a prioritized list
  for a human to open, not a content-generation trigger.
- Never treat `high_confidence_model` alone (p >= 0.7, no rule agreement) as sufficient to act
  without a human looking at the actual page -- high confidence is a property of the model's
  probability estimate, not a guarantee of being right, and Section 4's memorization gap means
  some of that confidence is client identity, not content signal.
- Never rank across clients as if scores were comparable in an absolute sense -- the model was
  never tested on a held-out *new* client, only on held-out rows from clients it partly learned
  from via the other folds.

In [5]:
recently_updated_in_t1 = t1["recently_updated"].sum()
print(f"tier 1 rows updated <14 days ago (recently_updated_flag) : {recently_updated_in_t1:,} of {len(t1):,}")

thin_clients = byc[byc < 30]
thin_rows = q_sorted[q_sorted["client_id"].isin(thin_clients.index)]
print(f"clients with <30 rows in this population : {len(thin_clients)} clients, {len(thin_rows):,} rows total")

outside_scope_flagged = q_sorted[
    q_sorted["reason_codes"].str.contains("outside_rule_scope") & (q_sorted["action_tier"] != "4_monitor")
]
print(f"outside-rule-scope rows the model still flagged for action : {len(outside_scope_flagged):,}")

tier 1 rows updated <14 days ago (recently_updated_flag) : 307 of 5,396
clients with <30 rows in this population : 4 clients, 82 rows total
outside-rule-scope rows the model still flagged for action : 356


## 6. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Snapshot values from this run, to compare future runs against:

- **Population base rate.** If a future run's base rate moves outside roughly +/-0.05 of this
  run's without a known cause (e.g. a new client added), that's a population-drift trigger --
  the tier thresholds (p >= 0.5, p >= 0.6, p >= 0.7) were picked against this base rate and may
  not mean the same thing against a different one.
- **Tier mix.** A large shift -- especially tier 2 growing much past its current share -- would
  mean the rule is going silent on more of the population than it did here, which is worth
  checking on its own rather than assuming the model is just finding more.
- **Tier-1 window-reliant share.** If this climbs, more of the "refresh now" tier is leaning on
  same-window columns (Section 4) rather than leading indicators -- re-run Week 6's Test E to
  confirm the AUC cost hasn't grown.
- **Retrain trigger.** Re-run the Week-6 grouped-vs-random audit whenever a new client is added
  or the feature set changes. If the gap moves materially past 0.122, that's a sign the model
  is leaning harder on client identity than it was here, and the "new client" caveat in
  Section 2 gets stronger, not weaker.
- **Outcome check (the only trigger that actually validates the queue).** For a sample of
  tier-1 and tier-2 pages a human refreshes, track whether their next 30/90-day trend actually
  improves. This queue has never been checked against a real outcome -- everything above is a
  measured association in one snapshot, not evidence the recommended actions work.

In [6]:
print(f"current population base rate (declining) : {y.mean():.3f}")
tier_mix = q_sorted["action_tier"].value_counts(normalize=True).sort_index()
print("current tier mix:")
print(tier_mix.round(3).to_string())
print(f"current tier-1 window-reliant share          : {t1['window_reliant'].mean():.3f}")

current population base rate (declining) : 0.611
current tier mix:
action_tier
1_refresh_now                0.203
2_investigate_rule_silent    0.424
3_ctr_gap_only               0.065
4_monitor                    0.308
current tier-1 window-reliant share          : 0.229


## 7. Cost / value thinking

*How far down this queue is worth an editor's time -- argued from measured quantities only.*

I am deliberately **not** attaching a currency figure to an editor-hour or a recovered click. I
don't have FlyRank's cost structure, and inventing one would dress a guess as an estimate. What
the data supports is a statement about **where the recoverable upside sits**, which is the part
of a cost/value argument that actually depends on evidence.

**Reach and recoverable upside are concentrated in a single archetype:**

| Archetype | Share of pages | Share of impressions | Share of recoverable missed clicks |
|---|---|---|---|
| **Visible performers** | **29.1%** | **87.5%** | **79.2%** |
| Deep low-demand | 31.4% | 6.2% | 10.1% |
| Young unproven | 29.6% | 5.8% | 9.7% |
| Low-visibility mid-age | 9.4% | 0.5% | 1.0% |
| Micro-volume | 0.5% | ~0% | ~0% |

("Recoverable missed clicks" = the Week-4 rule's positive `missed_clicks`, i.e. measured clicks
below tier-expected CTR. An observed gap, not a forecast of what a refresh would win back.)

**What follows, in cost/value terms:**

- **Editor effort is roughly constant per page; upside is not.** Opening a deep-low-demand or
  young-unproven page costs about what opening a visible-performer page costs, and reaches
  ~10% of the recoverable gap instead of 79.2%. That asymmetry -- not a dollar figure -- is the
  argument for reading the archetype column before the rank.
- **The two most expensive mistakes are both cheap to avoid.** Refreshing a deep-low-demand page
  spends editor time on a page with no demand to recover. Refreshing a young-unproven page spends
  it on a page whose downward trend may just be settling (Section 3). Both are avoided by
  checking the archetype first.
- **Tier 3 is cheap work and worth keeping.** The 1,730 `3_ctr_gap_only` rows carry a *measured*
  click gap with no model agreement. They need no model trust to justify -- the gap was counted,
  not predicted -- so they are low-risk even sitting below tier 2.
- **The number that would settle this does not exist here.** Nothing in this extract measures
  what a refresh recovers. Until the outcome check in Section 6 runs, every ratio above locates
  the *gap*; none of them price closing it. An ROI claim built on this queue today would be an
  assumption wearing a metric's clothing.

**Practical stopping rule:** work tier 1 and tier 2 *within the visible-performers archetype*
first. Below that, return per hour drops sharply for reasons the table shows directly. Stop when
the archetype column stops reading `protect` -- not at a fixed rank, and explicitly not by
sorting on age or update recency (Section 3).


In [7]:
print("=== where the recoverable upside actually sits ===")
print(archetype_profile[["n", "share_of_pages", "share_of_impressions",
                         "share_of_missed_clicks", "action"]].round(3).to_string())

_top = archetype_profile.loc[archetype_profile.index == "visible_performers"]
print(f"\nvisible_performers alone: {_top['share_of_pages'].sum():.1%} of pages, "
      f"{_top['share_of_impressions'].sum():.1%} of impressions, "
      f"{_top['share_of_missed_clicks'].sum():.1%} of recoverable missed clicks")

t3 = q_sorted[q_sorted["action_tier"] == "3_ctr_gap_only"]
print(f"tier 3 (measured CTR gap, no model agreement): {len(t3):,} rows, "
      f"median missed-click share {t3['baseline_score'].median():.3f}")
print("\nno currency figure appears above: this extract holds no refresh outcomes and no cost")
print("data, so return-per-hour cannot be estimated -- only located.")


=== where the recoverable upside actually sits ===
                          n  share_of_pages  share_of_impressions  share_of_missed_clicks          action
archetype                                                                                                
visible_performers     7740           0.291                 0.875                   0.792         protect
deep_low_demand        8358           0.314                 0.062                   0.101  merge_or_prune
young_unproven         7877           0.296                 0.058                   0.097            wait
low_visibility_midage  2489           0.094                 0.005                   0.010     investigate
micro_volume            140           0.005                 0.000                   0.000         exclude

visible_performers alone: 29.1% of pages, 87.5% of impressions, 79.2% of recoverable missed clicks
tier 3 (measured CTR gap, no model agreement): 1,730 rows, median missed-click share 0.442

no currency fig

## 8. Exports for the paper

*Queue -> `work/outputs/` (gitignored, regenerated). Metrics JSON -> `work/outputs/` (committed, the receipts). Figures -> `work/figures/` (committed, reused by the paper).*

In [8]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

export_cols = ["rank", "content_id", "client_id", "action_tier", "reason_codes",
               "archetype", "archetype_action",
               "baseline_score", "p_decline", "p_decline_strict", "position_tier",
               "impressions_90d", "clicks_90d", "avg_position", "content_age_days",
               "days_since_last_update", "truth_declining"]
q_sorted[export_cols].to_csv("work/outputs/w07_action_queue.csv", index=False)
print(f"wrote work/outputs/w07_action_queue.csv -- {len(q_sorted):,} rows")

metrics = {
    "population_rows": int(len(pop)),
    "base_rate_declining": round(float(y.mean()), 4),
    "n_clients": int(pop["client_id"].nunique()),
    "oof_auc_permissive": round(float(roc_auc_score(y, p_permissive)), 4),
    "oof_auc_strict": round(float(roc_auc_score(y, p_strict)), 4),
    "auc_note": "pooled out-of-fold across GroupKFold(5), not the per-fold mean quoted in w06",
    "action_tier_counts": {k: int(v) for k, v in q_sorted["action_tier"].value_counts().sort_index().items()},
    "archetypes": {
        "method": "KMeans k=5 on 9 scaled columns, log1p on heavy-tailed; metric clustering, NOT semantic",
        "excluded_feature": {
            "column": "days_since_last_update",
            "reason": "batch write timestamps: 57 distinct values, top 5 cover 87.8% of rows",
        },
        "silhouette_by_k": {str(k): round(v, 4) for k, v in sil_by_k.items()},
        "profiles": {
            str(name): {
                "n": int(r["n"]),
                "share_of_pages": round(float(r["share_of_pages"]), 4),
                "declining_rate": round(float(r["decline_rate"]), 4),
                "share_of_impressions": round(float(r["share_of_impressions"]), 4),
                "share_of_missed_clicks": round(float(r["share_of_missed_clicks"]), 4),
                "action": str(r["action"]),
            } for name, r in archetype_profile.iterrows()
        },
    },
    "age_and_staleness": {
        "declining_rate_by_age_tier": {str(k): round(float(v), 4)
                                       for k, v in age_decay["declining_rate"].items()},
        "point_biserial_r": decay_corr,
        "staleness_column_unusable": "days_since_last_update is batch-stamped; no staleness claim is drawn from it",
        "survivorship_caveat": "population requires impressions_prev_30d > 0; age effect cannot be separated from survivorship",
    },
}
with open("work/outputs/w07_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("wrote work/outputs/w07_playbook_metrics.json")

plt.rcParams.update({"figure.dpi": 130, "font.size": 9})

fig, ax = plt.subplots(figsize=(7.2, 3.4))
_p = archetype_profile.sort_values("share_of_missed_clicks")
_ypos = np.arange(len(_p))
ax.barh(_ypos - 0.2, _p["share_of_pages"], height=0.38, label="share of pages")
ax.barh(_ypos + 0.2, _p["share_of_missed_clicks"], height=0.38,
        label="share of recoverable missed clicks")
ax.set_yticks(_ypos); ax.set_yticklabels(_p.index)
ax.set_xlabel("share of population")
ax.set_title("Effort scales with pages; upside does not")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("work/figures/w07_archetype_value_concentration.svg")
plt.close(fig)

fig, ax = plt.subplots(figsize=(6.6, 3.2))
ax.plot(age_decay.index.astype(str), age_decay["declining_rate"], marker="o",
        label="by age tier (usable column)")
ax.plot(fresh_decay.index.astype(str), fresh_decay["declining_rate"], marker="s",
        linestyle=":", label="by freshness tier (batch-stamped, shown as artifact)")
ax.axhline(y.mean(), linestyle="--", linewidth=1, label=f"base rate ({y.mean():.3f})")
ax.set_ylabel("declining rate")
ax.set_title("Age separates the label; the update column cannot")
ax.legend(fontsize=7.5)
fig.tight_layout()
fig.savefig("work/figures/w07_age_vs_update_column.svg")
plt.close(fig)

print("wrote work/figures/w07_archetype_value_concentration.svg")
print("wrote work/figures/w07_age_vs_update_column.svg")


wrote work/outputs/w07_action_queue.csv -- 26,604 rows
wrote work/outputs/w07_playbook_metrics.json
wrote work/figures/w07_archetype_value_concentration.svg
wrote work/figures/w07_age_vs_update_column.svg


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.